# Fold 1: baseline frente a v1 (λ = 0.2) · copia mediante la API de Drive · ImageCAS1000 · 100 épocas

**Objetivo:** comprobar si el patrón observado en el fold 0 se repite en el fold 1. Con 1000 casos, v1 no supera al baseline en el fold 0; el fold 1 permite determinar si ese resultado es específico de aquella partición.

**Motivo de esta versión:** en una versión previa del notebook, la copia de los datos sobre el montaje FUSE de Drive (`shutil.copy2`) se bloqueó en cuatro ocasiones, siempre tras copiar aproximadamente 1.3 GB. Cuando el montaje FUSE se detiene a mitad de un archivo, `shutil.copy2` queda a la espera de forma indefinida y no lanza ninguna excepción, de modo que el bucle de reintentos nunca llega a actuar.

En esta versión, la transferencia de los 3000 archivos de `nnUNetPlans_3d_fullres` y de los 1000 de `gt_segmentations` se realiza con la API de Google Drive por HTTP (autenticación nativa de Colab) y con un tiempo de espera real de socket. Si una descarga se bloquea, se genera una excepción a los 90 s y el archivo se reintenta. Drive se sigue montando mediante FUSE únicamente para leer el split canónico (un archivo pequeño) y para escribir checkpoints y CSV de resultados (escrituras pequeñas e incrementales).

El resto del procedimiento (trainer v1, entrenamiento, evaluación y comparación pareada) es idéntico al de la versión anterior.

**Alcance de las salidas guardadas:** este notebook contiene la preparación de los datos y los dos entrenamientos completos (baseline y v1, 100 épocas cada uno). La evaluación sobre los 200 casos de validación quedó interrumpida en esta sesión y se completó en el notebook 32, que calcula las métricas sobre copias locales.


## 0. Comprobación de la GPU


In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Activa una GPU (A100 recomendado) antes de continuar.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")


PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.09 GB


## 1. Instalación de dependencias

Si el runtime procede de una sesión anterior con todo instalado, la celda es rápida y no es necesario reiniciar. En un runtime nuevo, se reinicia con la celda 1b tras la instalación.


In [1]:
!python -m pip install -q nnunetv2 medpy scikit-image SimpleITK
print("Dependencias instaladas.")


Dependencias instaladas.


## 1b. Reinicio del runtime (solo tras instalar en un runtime nuevo)

Si el runtime procede de la sesión anterior, esta celda puede omitirse.


In [ ]:
import os
os.kill(os.getpid(), 9)


## 2. Autenticación con la API de Drive (descargas pesadas) y montaje de Drive (resto de operaciones)

`auth.authenticate_user()` reutiliza el inicio de sesión de la cuenta de Colab, sin necesidad de introducir tokens. El tiempo de espera de socket (`socket.setdefaulttimeout(90)`) se activa después de la autenticación, ya que activarlo antes puede interferir con ella. Es el elemento que convierte cualquier descarga bloqueada en una excepción real a los 90 s.

Si la celda falla con `MessageError: credential propagation was unsuccessful`, se recarga la pestaña del navegador y se vuelve a ejecutar. Se trata de un fallo conocido del canal de comunicación entre el frontend y el kernel de Colab, no de un problema de la cuenta ni del notebook.


In [7]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

creds, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=creds)
print("✅ API de Drive autenticada.")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)
print("✅ Drive montado (solo para split/resultados, no para la copia pesada).")

# El tiempo de espera de socket se activa después de la autenticación: si se activa antes,
# puede interferir con el mecanismo de propagación de credenciales de Colab
# (error típico: "credential propagation was unsuccessful").
import socket
socket.setdefaulttimeout(90)
print("✅ Timeout de socket (90s) activado para las descargas.")


✅ API de Drive autenticada.
Mounted at /content/drive
✅ Drive montado (solo para split/resultados, no para la copia pesada).
✅ Timeout de socket (90s) activado para las descargas.


## 3. Definición de rutas y configuración (FOLD = 1)


In [2]:
import os, shutil, json, time, io
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

LOCAL_PREP_ROOT = Path("/content/nnUNet_preprocessed")
LOCAL_DATASET = LOCAL_PREP_ROOT / "Dataset503_ImageCAS1000"

NNUNET_RESULTS = BASE / "nnUNet_results"
CUSTOM_TRAINERS = BASE / "custom_trainers"
CUSTOM_TRAINERS.mkdir(parents=True, exist_ok=True)

DATASET_ID = 503
DATASET_FOLDER_NAME = "Dataset503_ImageCAS1000"
CONFIGURATION = "3d_fullres"
FOLD = 1

BASELINE_TRAINER = "nnUNetTrainer_100epochs"
V1_TRAINER = "nnUNetTrainerV1_1000casos_100epochs"
WEIGHT_DICE = 1.0
WEIGHT_CE = 1.0
LAMBDA_CLDICE = 0.2
SKELETON_ITERATIONS = 10

SPLIT_CORRECTO = BASE / "splits_final_Dataset503_1000_auto5fold.json"
assert SPLIT_CORRECTO.exists(), "Falta splits_final_Dataset503_1000_auto5fold.json en BASE."

print("Local dataset:", LOCAL_DATASET)
print("Resultados:", NNUNET_RESULTS)
print("Fold:", FOLD)


Local dataset: /content/nnUNet_preprocessed/Dataset503_ImageCAS1000
Resultados: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results
Fold: 1


## 4. Localización de las carpetas en Drive mediante la API y listado de sus archivos

Solo se consultan metadatos (nombre, identificador y tamaño). Esta celda no descarga ningún archivo, por lo que no puede bloquearse como la copia masiva.


In [3]:
def buscar_id(nombre, parent_id, carpeta=True):
    mime = " and mimeType = 'application/vnd.google-apps.folder'" if carpeta else ""
    q = f"name = '{nombre}' and '{parent_id}' in parents and trashed = false{mime}"
    res = drive_api.files().list(q=q, fields="files(id, name)", pageSize=5).execute()
    archivos = res.get("files", [])
    if not archivos:
        raise FileNotFoundError(f"No se encontro '{nombre}' dentro de {parent_id}")
    return archivos[0]["id"]

def listar_hijos(parent_id):
    archivos = []
    page_token = None
    while True:
        res = drive_api.files().list(
            q=f"'{parent_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, size, mimeType)",
            pageSize=1000,
            pageToken=page_token,
        ).execute()
        archivos.extend(res.get("files", []))
        page_token = res.get("nextPageToken")
        if not page_token:
            break
    return archivos

id_tfm = buscar_id("TFM_nnUNet", "root")
id_prep = buscar_id("nnUNet_preprocessed", id_tfm)
id_dataset = buscar_id("Dataset503_ImageCAS1000", id_prep)
id_cfg = buscar_id("nnUNetPlans_3d_fullres", id_dataset)
id_gt = buscar_id("gt_segmentations", id_dataset)

archivos_raiz_dataset = listar_hijos(id_dataset)
archivos_cfg = listar_hijos(id_cfg)
archivos_gt = listar_hijos(id_gt)

archivos_json_top = [f for f in archivos_raiz_dataset if f["name"] in
                      ("nnUNetPlans.json", "dataset.json", "dataset_fingerprint.json")]

print("JSON de nivel superior encontrados:", [f["name"] for f in archivos_json_top])
print("Archivos en nnUNetPlans_3d_fullres:", len(archivos_cfg))
print("Archivos en gt_segmentations:", len(archivos_gt))


JSON de nivel superior encontrados: ['nnUNetPlans.json', 'dataset.json', 'dataset_fingerprint.json']
Archivos en nnUNetPlans_3d_fullres: 3000
Archivos en gt_segmentations: 1000


## 5. Verificación de la integridad de Dataset503 en Drive (1000/1000, por metadatos)


In [4]:
imgs = [f for f in archivos_cfg if f["name"].startswith("ImageCAS_") and f["name"].endswith(".b2nd") and not f["name"].endswith("_seg.b2nd")]
segs = [f for f in archivos_cfg if f["name"].endswith("_seg.b2nd")]
pkls = [f for f in archivos_cfg if f["name"].endswith(".pkl")]
gts = [f for f in archivos_gt if f["name"].endswith(".nii.gz")]

print("Imagenes:", len(imgs), "Segmentaciones:", len(segs), "PKL:", len(pkls), "GT:", len(gts))
assert len(imgs) == 1000 and len(segs) == 1000 and len(pkls) == 1000 and len(gts) == 1000
assert len(archivos_json_top) == 3
print("\n✅ Dataset503 en Drive: 1000/1000 y JSON de planificacion presentes.")


Imagenes: 1000 Segmentaciones: 1000 PKL: 1000 GT: 1000

✅ Dataset503 en Drive: 1000/1000 y JSON de planificacion presentes.


## 6. Comprobación del espacio local


In [5]:
usage = shutil.disk_usage("/content")
free_gb = usage.free / 1024**3
print(f"Espacio libre local: {free_gb:.1f} GB")
if free_gb < 40:
    raise RuntimeError(f"Solo hay {free_gb:.1f} GB libres. Revisa antes de continuar.")
print("✅ Espacio local suficiente (o copia ya existente).")


Espacio libre local: 187.8 GB
✅ Espacio local suficiente (o copia ya existente).


## 7. Descarga robusta mediante la API (con tiempo de espera real)

Sustituye a la copia FUSE que se bloqueaba. Mantiene la capacidad de reanudación de la versión anterior (omite los archivos ya descargados con el tamaño correcto). Un bloqueo real se convierte ahora en una excepción, gracias al tiempo de espera de socket definido en la sección 2, de modo que el reintento automático puede actuar.


In [6]:
def descargar_lista(archivos_remotos, carpeta_local, max_reintentos=5):
    carpeta_local.mkdir(parents=True, exist_ok=True)
    print(f"Total de archivos a descargar: {len(archivos_remotos)}")

    pendientes = list(archivos_remotos)
    intento = 0
    while pendientes and intento < max_reintentos:
        intento += 1
        print(f"\n--- Intento {intento}/{max_reintentos}: {len(pendientes)} archivos pendientes ---")
        fallidos = []
        for i, meta in enumerate(pendientes):
            destino = carpeta_local / meta["name"]
            tam_esperado = int(meta.get("size", 0) or 0)
            try:
                if destino.exists() and (tam_esperado == 0 or destino.stat().st_size == tam_esperado):
                    continue
                request = drive_api.files().get_media(fileId=meta["id"])
                fh = io.FileIO(destino, "wb")
                downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024 * 20)
                done = False
                while not done:
                    status, done = downloader.next_chunk(num_retries=3)
                fh.close()
            except Exception as e:
                fallidos.append(meta)
                if len(fallidos) % 50 == 1:
                    print(f"  fallo en {meta['name']}: {type(e).__name__}: {e}")
            if i % 100 == 0:
                print(f"  {i}/{len(pendientes)} procesados ({len(fallidos)} fallos hasta ahora)")

        pendientes = fallidos
        if pendientes:
            print(f"Quedan {len(pendientes)} archivos fallidos. Esperando 15s antes de reintentar...")
            time.sleep(15)

    if pendientes:
        print(f"⚠️ Tras {max_reintentos} intentos siguen fallando {len(pendientes)} archivos.")
    else:
        print("✅ Descarga completa (verificada por tamano).")
    return pendientes


if LOCAL_DATASET.exists():
    print("Ya existe una copia local -- se reutiliza y se completa lo que falte (no se borra).")
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

pend_json = descargar_lista(archivos_json_top, LOCAL_DATASET)
assert not pend_json, "Fallo al descargar los JSON de nivel superior."

cfg_local = LOCAL_DATASET / "nnUNetPlans_3d_fullres"
gt_local = LOCAL_DATASET / "gt_segmentations"

t0 = time.time()
pend_cfg = descargar_lista(archivos_cfg, cfg_local)
print(f"nnUNetPlans_3d_fullres: {(time.time()-t0)/60:.1f} min, {len(pend_cfg)} archivos sin descargar")
assert not pend_cfg, "Quedaron archivos sin descargar en nnUNetPlans_3d_fullres."

t0 = time.time()
pend_gt = descargar_lista(archivos_gt, gt_local)
print(f"gt_segmentations: {(time.time()-t0)/60:.1f} min, {len(pend_gt)} archivos sin descargar")
assert not pend_gt, "Quedaron archivos sin descargar en gt_segmentations."

shutil.copy2(SPLIT_CORRECTO, LOCAL_DATASET / "splits_final.json")
print("\n✅ Copia local lista (via API de Drive), incluido el split canonico (5 folds).")


Total de archivos a descargar: 3

--- Intento 1/5: 3 archivos pendientes ---
  0/3 procesados (0 fallos hasta ahora)
✅ Descarga completa (verificada por tamano).
Total de archivos a descargar: 3000

--- Intento 1/5: 3000 archivos pendientes ---
  0/3000 procesados (0 fallos hasta ahora)
  100/3000 procesados (0 fallos hasta ahora)
  200/3000 procesados (0 fallos hasta ahora)
  300/3000 procesados (0 fallos hasta ahora)
  400/3000 procesados (0 fallos hasta ahora)


  500/3000 procesados (0 fallos hasta ahora)
  600/3000 procesados (0 fallos hasta ahora)
  700/3000 procesados (0 fallos hasta ahora)
  800/3000 procesados (0 fallos hasta ahora)
  900/3000 procesados (0 fallos hasta ahora)


  1000/3000 procesados (0 fallos hasta ahora)
  1100/3000 procesados (0 fallos hasta ahora)
  1200/3000 procesados (0 fallos hasta ahora)
  1300/3000 procesados (0 fallos hasta ahora)
  1400/3000 procesados (0 fallos hasta ahora)


  1500/3000 procesados (0 fallos hasta ahora)
  1600/3000 procesados (0 fallos hasta ahora)
  1700/3000 procesados (0 fallos hasta ahora)
  1800/3000 procesados (0 fallos hasta ahora)
  1900/3000 procesados (0 fallos hasta ahora)


  2000/3000 procesados (0 fallos hasta ahora)
  2100/3000 procesados (0 fallos hasta ahora)
  2200/3000 procesados (0 fallos hasta ahora)
  2300/3000 procesados (0 fallos hasta ahora)
  2400/3000 procesados (0 fallos hasta ahora)
  2500/3000 procesados (0 fallos hasta ahora)


  2600/3000 procesados (0 fallos hasta ahora)
  2700/3000 procesados (0 fallos hasta ahora)
  2800/3000 procesados (0 fallos hasta ahora)
  2900/3000 procesados (0 fallos hasta ahora)
✅ Descarga completa (verificada por tamano).
nnUNetPlans_3d_fullres: 334.0 min, 0 archivos sin descargar
Total de archivos a descargar: 1000

--- Intento 1/5: 1000 archivos pendientes ---
  0/1000 procesados (0 fallos hasta ahora)


  100/1000 procesados (0 fallos hasta ahora)
  200/1000 procesados (0 fallos hasta ahora)


  300/1000 procesados (0 fallos hasta ahora)
  400/1000 procesados (0 fallos hasta ahora)
  500/1000 procesados (0 fallos hasta ahora)


  600/1000 procesados (0 fallos hasta ahora)
  700/1000 procesados (0 fallos hasta ahora)


  800/1000 procesados (0 fallos hasta ahora)
  900/1000 procesados (0 fallos hasta ahora)
✅ Descarga completa (verificada por tamano).
gt_segmentations: 223.8 min, 0 archivos sin descargar

✅ Copia local lista (via API de Drive), incluido el split canonico (5 folds).


## 8. Verificación de la copia local (1000/1000), del split canónico y del tamaño del fold 1


In [7]:
local_cfg = LOCAL_DATASET / "nnUNetPlans_3d_fullres"
local_gt = LOCAL_DATASET / "gt_segmentations"

imgs_local = [p for p in local_cfg.glob("ImageCAS_*.b2nd") if not p.name.endswith("_seg.b2nd")]
segs_local = list(local_cfg.glob("ImageCAS_*_seg.b2nd"))
pkls_local = list(local_cfg.glob("ImageCAS_*.pkl"))
gts_local = list(local_gt.glob("ImageCAS_*.nii.gz"))

print("Imagenes:", len(imgs_local), "Segmentaciones:", len(segs_local), "PKL:", len(pkls_local), "GT:", len(gts_local))
assert len(imgs_local) == 1000 and len(segs_local) == 1000 and len(pkls_local) == 1000 and len(gts_local) == 1000

with open(LOCAL_DATASET / "splits_final.json") as f:
    split_local = json.load(f)
with open(SPLIT_CORRECTO) as f:
    split_canonico = json.load(f)
assert split_local == split_canonico, "El split local NO coincide con el canonico."
assert len(split_local) >= 2, "El split no tiene al menos 2 folds."

fold1 = split_local[FOLD]
print(f"Fold {FOLD}: {len(fold1['train'])} train / {len(fold1['val'])} val")
assert len(fold1["train"]) == 800 and len(fold1["val"]) == 200

print("\n✅ Copia local integra y split canonico confirmado para fold", FOLD)


Imagenes: 1000 Segmentaciones: 1000 PKL: 1000 GT: 1000
Fold 1: 800 train / 200 val

✅ Copia local integra y split canonico confirmado para fold 1


## 9. Configuración de nnU-Net: datos en el SSD local y resultados en Drive

La celda siguiente redefine `GT_FOLDER` para que apunte al ground truth almacenado en Drive, que es la ruta utilizada por las funciones de evaluación.


In [3]:
os.environ["nnUNet_preprocessed"] = str(LOCAL_PREP_ROOT)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)
NNUNET_RAW = BASE / "nnUNet_raw"
os.environ["nnUNet_raw"] = str(NNUNET_RAW)

GT_FOLDER = LOCAL_DATASET / "gt_segmentations"

print("nnUNet_preprocessed =", os.environ["nnUNet_preprocessed"])
print("nnUNet_results      =", os.environ["nnUNet_results"])
print("nnUNet_extTrainer   =", os.environ["nnUNet_extTrainer"])


nnUNet_preprocessed = /content/nnUNet_preprocessed
nnUNet_results      = /content/drive/MyDrive/TFM_nnUNet/nnUNet_results
nnUNet_extTrainer   = /content/drive/MyDrive/TFM_nnUNet/custom_trainers


In [4]:
GT_FOLDER = BASE / "nnUNet_preprocessed" / "Dataset503_ImageCAS1000" / "gt_segmentations"
print("GT_FOLDER (Drive):", GT_FOLDER)

GT_FOLDER (Drive): /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000/gt_segmentations


## 10. Funciones de métricas (idénticas a las de los notebooks 28 y 29, para garantizar la comparabilidad)

La celda posterior redefine `evaluar_trainer` incorporando un checkpoint local que permite reanudar el cálculo si el kernel se interrumpe.


In [6]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)

def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing

def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)

def hard_cldice(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    return (2 * tprec * tsens) / (tprec + tsens + 1e-8)

def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n

def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n

def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)

def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)
    return {
        "dice": dice_score(pred, gt),
        "cldice": hard_cldice(pred, gt),
        "cobertura_principal": cobertura,
        "n_componentes": n_componentes,
        "hd95_crudo_mm": hd95_seguro(pred, gt, spacing),
    }

def evaluar_trainer(trainer_name):
    carpeta_val = (
        NNUNET_RESULTS / DATASET_FOLDER_NAME
        / f"{trainer_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}" / "validation"
    )
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    filas = []
    for pred_path in preds:
        case_id = pred_path.name.replace(".nii.gz", "")
        gt_path = GT_FOLDER / pred_path.name
        if not gt_path.exists():
            continue
        m = calcular_metricas_caso(pred_path, gt_path)
        m["case_id"] = case_id
        filas.append(m)

    df = pd.DataFrame(filas)
    resumen = {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }
    return df, resumen

print("Funciones de metricas cargadas.")


Funciones de metricas cargadas.


In [9]:
def evaluar_trainer(trainer_name, guardar_cada=20):
    carpeta_val = (
        NNUNET_RESULTS / DATASET_FOLDER_NAME
        / f"{trainer_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}" / "validation"
    )
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    # Checkpoint local para poder reanudar si el kernel se interrumpe a mitad
    checkpoint_path = Path(f"/content/checkpoint_{trainer_name}.csv")

    filas = []
    ya_hechos = set()
    if checkpoint_path.exists():
        df_prev = pd.read_csv(checkpoint_path)
        filas = df_prev.to_dict("records")
        ya_hechos = set(df_prev["case_id"])
        print(f"Reanudando: {len(ya_hechos)} casos ya calculados antes, se saltan.")

    total = len(preds)
    for i, pred_path in enumerate(preds):
        case_id = pred_path.name.replace(".nii.gz", "")
        if case_id in ya_hechos:
            continue

        gt_path = GT_FOLDER / pred_path.name
        if not gt_path.exists():
            continue

        m = calcular_metricas_caso(pred_path, gt_path)
        m["case_id"] = case_id
        filas.append(m)

        if (i + 1) % guardar_cada == 0 or (i + 1) == total:
            pd.DataFrame(filas).to_csv(checkpoint_path, index=False)
            print(f"[{trainer_name}] {i + 1}/{total} procesados -- checkpoint guardado")

    df = pd.DataFrame(filas)
    resumen = {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }
    return df, resumen

## 11. Trainer v1 (idempotente)

Es el mismo trainer del notebook 28: Dice + CE + 0.2 · soft-clDice, con `DeepSupervisionWrapper` envolviendo el módulo combinado completo.


In [10]:
TRAINER_TEMPLATE = r'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_{epochs}epochs
)


def soft_erode_3d(x):
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)
    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


class SoftClDiceLoss(nn.Module):
    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()
        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)
        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):
    def __init__(self, regional_loss, lambda_cldice=0.2, skeleton_iterations=10):
        super().__init__()
        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice
        self.soft_cldice = SoftClDiceLoss(iterations=skeleton_iterations, smooth=1e-5)

    def forward(self, logits, target):
        loss_regional = self.regional_loss(logits, target)

        probabilities = torch.softmax(logits.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]

        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)
        foreground_target = (target == 1).float()

        topology_loss = self.soft_cldice(foreground_probability, foreground_target)

        return loss_regional + self.lambda_cldice * topology_loss


class {class_name}(nnUNetTrainer_{epochs}epochs):

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.weight_dice = {weight_dice}
        self.weight_ce = {weight_ce}
        self.lambda_cldice = {lambda_cldice}
        self.skeleton_iterations = {skeleton_iterations}

    def on_train_start(self):
        super().on_train_start()
        self.print_to_log_file(
            f"[v1 fold {{self.fold}}] weight_dice={{self.weight_dice}} "
            f"weight_ce={{self.weight_ce}} lambda_cldice={{self.lambda_cldice}} "
            f"skeleton_iterations={{self.skeleton_iterations}}"
        )

    def _build_loss(self):
        regional_loss = DC_and_CE_loss(
            soft_dice_kwargs={{
                "batch_dice": self.configuration_manager.batch_dice,
                "smooth": 1e-5,
                "do_bg": False,
                "ddp": self.is_ddp
            }},
            ce_kwargs={{}},
            weight_ce=self.weight_ce,
            weight_dice=self.weight_dice,
            ignore_label=self.label_manager.ignore_label,
            dice_class=MemoryEfficientSoftDiceLoss
        )

        loss = DiceCEPlusSoftClDice(
            regional_loss=regional_loss,
            lambda_cldice=self.lambda_cldice,
            skeleton_iterations=self.skeleton_iterations,
        )

        if self.enable_deep_supervision:
            deep_supervision_scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** i) for i in range(len(deep_supervision_scales))])
            weights[-1] = 0
            weights = weights / weights.sum()
            loss = DeepSupervisionWrapper(loss, weights)

        return loss
'''

codigo = TRAINER_TEMPLATE.format(
    class_name=V1_TRAINER,
    epochs=100,
    weight_dice=WEIGHT_DICE,
    weight_ce=WEIGHT_CE,
    lambda_cldice=LAMBDA_CLDICE,
    skeleton_iterations=SKELETON_ITERATIONS,
)
ruta_trainer = CUSTOM_TRAINERS / f"{V1_TRAINER}.py"
ruta_trainer.write_text(codigo)
print("Trainer escrito en:", ruta_trainer)


Trainer escrito en: /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerV1_1000casos_100epochs.py


## 12. Comprobación final previa al entrenamiento


In [11]:
print("=== CHECK FINAL (fold", FOLD, ") ===")
print("Baseline trainer:", BASELINE_TRAINER, "(built-in)")
print("v1 trainer:", V1_TRAINER, "->", (CUSTOM_TRAINERS / f"{V1_TRAINER}.py").exists())
print("Imagenes locales:", len(imgs_local), "| Split == canonico:", split_local == split_canonico)

for trainer in [BASELINE_TRAINER, V1_TRAINER]:
    posible_previo = NNUNET_RESULTS / DATASET_FOLDER_NAME / f"{trainer}__nnUNetPlans__{CONFIGURATION}" / f"fold_{FOLD}"
    if posible_previo.exists():
        print(f"\n⚠️ Ya existe resultado previo para {trainer} en fold {FOLD}: {posible_previo}")
    else:
        print(f"\nNo hay entrenamiento previo de {trainer} en fold {FOLD} -- listo para lanzar.")

assert len(imgs_local) == 1000
assert split_local == split_canonico
print("\n✅ LISTO PARA ENTRENAR FOLD", FOLD)


=== CHECK FINAL (fold 1 ) ===
Baseline trainer: nnUNetTrainer_100epochs (built-in)
v1 trainer: nnUNetTrainerV1_1000casos_100epochs -> True
Imagenes locales: 1000 | Split == canonico: True

⚠️ Ya existe resultado previo para nnUNetTrainer_100epochs en fold 1: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results/Dataset503_ImageCAS1000/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_1

No hay entrenamiento previo de nnUNetTrainerV1_1000casos_100epochs en fold 1 -- listo para lanzar.

✅ LISTO PARA ENTRENAR FOLD 1


## 13. Entrenamiento 1/2 — baseline, fold 1, 100 épocas

Las celdas posteriores se ejecutan una vez finalizado el entrenamiento. La validación final del baseline alcanza un Dice medio de 0.7541.


In [12]:
!nnUNetv2_train 503 3d_fullres 1 -tr nnUNetTrainer_100epochs --npz



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-09-22 00:30:34.910469: Using torch.compile...
2026-09-22 00:30:38.099786: do_dummy_2d_data_aug: False
2026-09-22 00:30:38.104826: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset503_ImageCAS1000/splits_final.json
2026-09-22 00:30:38.107257: The split file contains

## 14. Entrenamiento 2/2 — v1 (λ = 0.2), fold 1, 100 épocas

Las celdas posteriores se ejecutan una vez finalizado el entrenamiento. El aviso inicial `Trainer ... not found in nnunetv2.training.nnUNetTrainer` es el comportamiento habitual de nnU-Net al buscar primero en su paquete; a continuación, el trainer se localiza en la carpeta de trainers externos.


In [ ]:
!nnUNetv2_train 503 3d_fullres 1 -tr nnUNetTrainerV1_1000casos_100epochs --npz



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Trainer 'nnUNetTrainerV1_1000casos_100epochs' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Searching for class nnUNetTrainerV1_1000casos_100epochs in folder /content/drive/MyDrive/TFM_nnUNet/custom_trainers with current module None
  Inspecting module: nnUNetTrainerLambdaV1Fiel_100epochs
  Inspecting module: nnUNetTrainerOptunaBest_100epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial0_50epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial1_50epochs
  Inspecting module: nnUNetTrainerOptunaLambdaOnly_trial2_50epochs
  Inspecting module: nnUN

## 14b. Desconexión de Colab durante cualquiera de los dos entrenamientos

Si `/content` conserva la copia local intacta, se relanza con `--c` el entrenamiento interrumpido:


In [ ]:
# Ejemplo para el entrenamiento interrumpido (se descomenta la línea correspondiente):
#!nnUNetv2_train 503 3d_fullres 1 -tr nnUNetTrainer_100epochs --npz --c
#!nnUNetv2_train 503 3d_fullres 1 -tr nnUNetTrainerV1_1000casos_100epochs --npz --c

# Si se perdió el runtime completo, se vuelve a ejecutar desde la sección 2;
# la descarga de la sección 7 omite los archivos que sigan intactos.


## 15. Evaluación de ambos modelos en el fold 1 y guardado de métricas caso a caso

La evaluación quedó interrumpida en esta sesión tras 60 de los 200 casos del baseline. Se completó en el notebook 32, donde las predicciones y el ground truth se descargan a disco local antes de calcular las métricas.


In [ ]:
df_base_f1, resumen_base_f1 = evaluar_trainer(BASELINE_TRAINER)
print("=== Baseline (Dice+CE), fold 1, 100 epocas ===")
print(f"Dice: {resumen_base_f1['dice_mean']:.4f} | clDice: {resumen_base_f1['cldice_mean']:.4f} | "
      f"n_componentes: {resumen_base_f1['n_componentes_mean']:.2f} | HD95: {resumen_base_f1['hd95_crudo_mean']:.2f} mm | "
      f"n={resumen_base_f1['n_casos']}")

csv_base_f1 = BASE / "baseline_1000casos_fold1_100epochs_metricas_por_caso.csv"
df_base_f1.to_csv(csv_base_f1, index=False)
print("Guardado:", csv_base_f1)

df_v1_f1, resumen_v1_f1 = evaluar_trainer(V1_TRAINER)
print("\n=== v1 λ=.2, fold 1, 100 epocas ===")
print(f"Dice: {resumen_v1_f1['dice_mean']:.4f} | clDice: {resumen_v1_f1['cldice_mean']:.4f} | "
      f"n_componentes: {resumen_v1_f1['n_componentes_mean']:.2f} | HD95: {resumen_v1_f1['hd95_crudo_mean']:.2f} mm | "
      f"n={resumen_v1_f1['n_casos']}")

csv_v1_f1 = BASE / "v1_1000casos_fold1_100epochs_metricas_por_caso.csv"
df_v1_f1.to_csv(csv_v1_f1, index=False)
print("Guardado:", csv_v1_f1)


[nnUNetTrainer_100epochs] 20/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 40/200 procesados -- checkpoint guardado
[nnUNetTrainer_100epochs] 60/200 procesados -- checkpoint guardado


In [ ]:
print("sigue vivo?")

## 16. Comparación pareada en el fold 1 (mismos pacientes de validación)

Celda no ejecutada en esta sesión. Los resultados de la comparación pareada del fold 1 se encuentran en el notebook 32.


In [ ]:
from scipy import stats

df_base_f1s = df_base_f1.sort_values("case_id").reset_index(drop=True)
df_v1_f1s = df_v1_f1.sort_values("case_id").reset_index(drop=True)

assert list(df_base_f1s["case_id"]) == list(df_v1_f1s["case_id"]), \
    "case_id no coinciden entre baseline y v1 en fold 1 -- revisa el split."

delta_componentes_f1 = df_base_f1s["n_componentes"] - df_v1_f1s["n_componentes"]

resumen_tabla_f1 = pd.DataFrame([
    {"modelo": "baseline (fold 1)", "dice": resumen_base_f1["dice_mean"], "cldice": resumen_base_f1["cldice_mean"],
     "n_componentes": resumen_base_f1["n_componentes_mean"], "hd95_mm": resumen_base_f1["hd95_crudo_mean"]},
    {"modelo": "v1 λ=.2 (fold 1)", "dice": resumen_v1_f1["dice_mean"], "cldice": resumen_v1_f1["cldice_mean"],
     "n_componentes": resumen_v1_f1["n_componentes_mean"], "hd95_mm": resumen_v1_f1["hd95_crudo_mean"]},
])
print(resumen_tabla_f1.to_string(index=False))

print(f"\nΔcomponentes medio (baseline - v1), fold 1 = {delta_componentes_f1.mean():.3f}")
print("(positivo => v1 tiene MENOS componentes que baseline, apoyando H1)")

print("\n=== Tests pareados fold 1 (mismos 200 pacientes) ===")
for metrica, nombre in [("dice","Dice"), ("cldice","clDice"), ("n_componentes","n_componentes"), ("hd95_crudo_mm","HD95")]:
    a = df_base_f1s[metrica].values
    b = df_v1_f1s[metrica].values
    mask = ~(np.isnan(a) | np.isnan(b))
    w_stat, w_p = stats.wilcoxon(a[mask], b[mask])
    t_stat, t_p = stats.ttest_rel(a[mask], b[mask])
    print(f"{nombre}: Wilcoxon p={w_p:.4g} | ttest_rel p={t_p:.4g} (n={mask.sum()})")

csv_delta_f1 = BASE / "delta_componentes_1000casos_fold1_baseline_vs_v1.csv"
pd.DataFrame({
    "case_id": df_base_f1s["case_id"],
    "n_componentes_baseline": df_base_f1s["n_componentes"],
    "n_componentes_v1": df_v1_f1s["n_componentes"],
    "delta_componentes": delta_componentes_f1,
}).to_csv(csv_delta_f1, index=False)
print("\nGuardado:", csv_delta_f1)


## 17. Consistencia entre el fold 0 y el fold 1

Se cargan los resultados del fold 0 (calculados en el notebook 29) y se contrastan con los del fold 1. Celda no ejecutada en esta sesión; el resultado se encuentra en el notebook 32.


In [ ]:
df_base_f0 = pd.read_csv(BASE / "baseline_1000casos_100epochs_metricas_por_caso.csv")
df_v1_f0 = pd.read_csv(BASE / "v1_1000casos_100epochs_metricas_por_caso.csv")

comparacion_folds = pd.DataFrame([
    {"fold": 0, "modelo": "baseline", "dice": df_base_f0["dice"].mean(), "cldice": df_base_f0["cldice"].mean(),
     "n_componentes": df_base_f0["n_componentes"].mean(), "hd95_mm": df_base_f0["hd95_crudo_mm"].mean()},
    {"fold": 0, "modelo": "v1 λ=.2", "dice": df_v1_f0["dice"].mean(), "cldice": df_v1_f0["cldice"].mean(),
     "n_componentes": df_v1_f0["n_componentes"].mean(), "hd95_mm": df_v1_f0["hd95_crudo_mm"].mean()},
    {"fold": 1, "modelo": "baseline", "dice": resumen_base_f1["dice_mean"], "cldice": resumen_base_f1["cldice_mean"],
     "n_componentes": resumen_base_f1["n_componentes_mean"], "hd95_mm": resumen_base_f1["hd95_crudo_mean"]},
    {"fold": 1, "modelo": "v1 λ=.2", "dice": resumen_v1_f1["dice_mean"], "cldice": resumen_v1_f1["cldice_mean"],
     "n_componentes": resumen_v1_f1["n_componentes_mean"], "hd95_mm": resumen_v1_f1["hd95_crudo_mean"]},
])
print(comparacion_folds.to_string(index=False))

csv_folds = BASE / "comparacion_fold0_vs_fold1_baseline_vs_v1.csv"
comparacion_folds.to_csv(csv_folds, index=False)
print("\nGuardado:", csv_folds)


## Archivos persistentes

- Checkpoints y resultados → `MyDrive/TFM_nnUNet/nnUNet_results`
- Métricas del fold 1 → `baseline_1000casos_fold1_100epochs_metricas_por_caso.csv`, `v1_1000casos_fold1_100epochs_metricas_por_caso.csv`
- Δ componentes del fold 1 → `delta_componentes_1000casos_fold1_baseline_vs_v1.csv`
- Comparación fold 0 frente a fold 1 → `comparacion_fold0_vs_fold1_baseline_vs_v1.csv`
